In [ ]:
import cv2
import numpy as np
import os
import pandas as pd
import math
import time


In [ ]:
# Paths
VIDEO_PATH = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\megadataset\video12_2.mp4"
K_NEW_PATH = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\newK.npy"
MAP1_PATH  = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\map1.npy"
MAP2_PATH  = r"C:\AAKASH\MS_NOTES\THESIS\Material\calibration_webcam\calrecordings_newsetup3\results\map2.npy"

OUTPUT_FOLDER = r"C:\AAKASH\MS_NOTES\THESIS\Material\Data\recordings\megadataset_output_non_batch"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_FOLDER, "tracking_data.csv")


# Physical dimensions in mm
FIELD_WIDTH_MM  = 1200
FIELD_HEIGHT_MM = 680
PAD             = 50
H_PLATFORM     = -118.0
H_ROD           = -85.0
BORDER_WIDTH    = 9.5


# virtual dimensions in mm
HIT_ZONE_WIDTH  = 120
HIT_ZONE_HEIGHT = 80
GOAL_Y_CENTER   = FIELD_HEIGHT_MM / 2
GOAL_TOLERANCE  = 100


#Color threshold
LOWER_RED = np.array([152, 101, 175], dtype=np.uint8)
UPPER_RED = np.array([175, 255, 255], dtype=np.uint8)


# Filter parameters
BILATERAL_D           = 9
BILATERAL_SIGMA_COLOR = 30
BILATERAL_SIGMA_SPACE = 100
MIN_MOVEMENT_MM       = 3

# Hit detection thresholds
STOPPED_THRESHOLD    = 0.10   
SLOW_EXIT_THRESHOLD  = 0.30   
FAST_EXIT_THRESHOLD  = 0.50   
MIN_DIRECTION_CHANGE = 25     
BLOCK_ANGLE          = 150    
MIN_FRAMES_CONTROL   = 3      


In [ ]:
# Physical conifiguration of Rods
ROD_CONFIG = {
    1: {'x': 75,   'team': 'Black', 'role': 'GK',  'num_players': 1, 'player_color': 'dark' ,'offsets_bottom': None, 'offsets_top': None},
    2: {'x': 225,  'team': 'Black', 'role': 'DEF', 'num_players': 2, 'player_color': 'dark' ,'offsets_bottom': [15, -219], 'offsets_top': [-15, 219]},
    3: {'x': 375,  'team': 'White', 'role': 'ATK', 'num_players': 3, 'player_color': 'light','offsets_bottom': [-15, -199, -383], 'offsets_top': [15, 199, 383]},
    4: {'x': 525,  'team': 'Black', 'role': 'MID', 'num_players': 5, 'player_color': 'dark' ,'offsets_bottom': [15, -106.5, -228, -349.5, -471], 'offsets_top': [-15, 106.5, 228, 349.5, 471]},
    5: {'x': 675,  'team': 'White', 'role': 'MID', 'num_players': 5, 'player_color': 'light','offsets_bottom': [-15, -136.5, -258, -379.5, -501], 'offsets_top': [15, 136.5, 258, 379.5, 501]},
    6: {'x': 825,  'team': 'Black', 'role': 'ATK', 'num_players': 3, 'player_color': 'dark' ,'offsets_bottom': [15, -169, -353], 'offsets_top': [-15, 169, 353]},
    7: {'x': 975,  'team': 'White', 'role': 'DEF', 'num_players': 2, 'player_color': 'light','offsets_bottom': [-15, -249], 'offsets_top': [15, 249]},
    8: {'x': 1125, 'team': 'White', 'role': 'GK',  'num_players': 1, 'player_color': 'light','offsets_bottom': None, 'offsets_top': None},
}

In [ ]:
# Coordinate system
qr_world_points = np.array([
    [-BORDER_WIDTH, BORDER_WIDTH, H_PLATFORM],
    [FIELD_WIDTH_MM + BORDER_WIDTH, BORDER_WIDTH, H_PLATFORM],
    [FIELD_WIDTH_MM + BORDER_WIDTH, FIELD_HEIGHT_MM - BORDER_WIDTH, H_PLATFORM],
    [-BORDER_WIDTH, FIELD_HEIGHT_MM - BORDER_WIDTH, H_PLATFORM]
], dtype=np.float32)

field_world_points = np.array([
    [0, 0, 0], [FIELD_WIDTH_MM, 0, 0],
    [FIELD_WIDTH_MM, FIELD_HEIGHT_MM, 0], [0, FIELD_HEIGHT_MM, 0]
], dtype=np.float32)

dst_points = np.array([
    [PAD, PAD], [FIELD_WIDTH_MM + PAD, PAD],
    [FIELD_WIDTH_MM + PAD, FIELD_HEIGHT_MM + PAD], [PAD, FIELD_HEIGHT_MM + PAD]
], dtype=np.float32)


In [ ]:
def get_marker_data(img):
    aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    detector   = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())
    corners, ids, _ = detector.detectMarkers(img)
    return corners, ids

def compute_homography_and_pose(img_undist, K_new):
    corners, ids = get_marker_data(img_undist)
    if ids is None:
        return None, None, None
    marker_dict = {int(ids[i][0]): corners[i].reshape(4, 2) for i in range(len(ids))}
    if not all(m in marker_dict for m in [0, 1, 2, 3]):
        return None, None, None
    qr_image_points = np.array([
        marker_dict[0][2], marker_dict[1][3],
        marker_dict[2][0], marker_dict[3][1]
    ], dtype=np.float32)
    success, rvec, tvec = cv2.solvePnP(qr_world_points, qr_image_points, K_new, None, flags=cv2.SOLVEPNP_ITERATIVE)
    if not success:
        return None, None, None
    rvec, tvec = cv2.solvePnPRefineLM(qr_world_points, qr_image_points, K_new, None, rvec, tvec)
    field_corners_img, _ = cv2.projectPoints(field_world_points, rvec, tvec, K_new, None)
    H, _ = cv2.findHomography(field_corners_img.reshape(-1, 2), dst_points)
    return H, rvec, tvec

def project_rod_line_to_warped(rod_x, rvec, tvec, K_new, H, num_samples=200):
    y_positions = np.linspace(0, FIELD_HEIGHT_MM, num_samples)
    rod_3d = np.array([[rod_x, y, H_ROD] for y in y_positions], dtype=np.float32)
    rod_img, _ = cv2.projectPoints(rod_3d, rvec, tvec, K_new, None)
    warped_pts = []
    for pt in rod_img.reshape(-1, 2):
        wpt = H @ np.array([pt[0], pt[1], 1.0])
        warped_pts.append((int(wpt[0]/wpt[2]), int(wpt[1]/wpt[2])))
    return warped_pts, y_positions

def extract_intensities_along_line(image, points):
    h, w = image.shape[:2]
    intensities = []
    for (x, y) in points:
        x, y = max(0, min(w-1, x)), max(0, min(h-1, y))
        if len(image.shape) == 3:
            bgr = image[y, x]
            intensities.append(0.299*bgr[2] + 0.587*bgr[1] + 0.114*bgr[0])
    return np.array(intensities)

def apply_bilateral_filter_1d(intensities):
    signal_2d = intensities.reshape(1, -1).astype(np.float32)
    return cv2.bilateralFilter(signal_2d, BILATERAL_D, BILATERAL_SIGMA_COLOR, BILATERAL_SIGMA_SPACE).flatten()

stable_positions = {r: None for r in range(1, 9)}

def smart_stabilize(rod_num, new_pos):
    global stable_positions
    if len(new_pos) == 0:
        return stable_positions[rod_num] if stable_positions[rod_num] else []
    if stable_positions[rod_num] is None or len(stable_positions[rod_num]) != len(new_pos):
        stable_positions[rod_num] = list(new_pos)
        return new_pos
    if max(abs(new_pos[i] - stable_positions[rod_num][i]) for i in range(len(new_pos))) <= MIN_MOVEMENT_MM:
        return stable_positions[rod_num]
    stable_positions[rod_num] = list(new_pos)
    return new_pos

def reset_stabilization():
    global stable_positions
    stable_positions = {r: None for r in range(1, 9)}

def get_player_id(rod_num, player_idx):
    return f"{rod_num}.{player_idx + 1}"

def check_ball_in_hit_zone(bx, by, rx, py):
    if bx is None or by is None:
        return False
    return ((rx - HIT_ZONE_WIDTH/2) <= bx <= (rx + HIT_ZONE_WIDTH/2) and (py - HIT_ZONE_HEIGHT/2) <= by <= (py + HIT_ZONE_HEIGHT/2))

def find_ball_in_hit_zones(bx, by, all_pos):
    if bx is None or by is None:
        return "None"
    for rod_num, positions in all_pos.items():
        rx = ROD_CONFIG[rod_num]['x']
        for idx, py in enumerate(positions):
            if check_ball_in_hit_zone(bx, by, rx, py):
                return get_player_id(rod_num, idx)
    return "None"

print("Functions defined.")

In [ ]:
def find_peaks_above_threshold(inverted, y_positions, threshold):
    is_peak = inverted > threshold
    diff    = np.diff(is_peak.astype(int), prepend=0, append=0)
    starts  = np.where(diff == 1)[0]
    ends    = np.where(diff == -1)[0]
    regions = []
    for s, e in zip(starts, ends):
        s, e = max(0, s), min(len(y_positions)-1, e)
        regions.append({
            'y_start':  y_positions[s],
            'y_end':    y_positions[e],
            'y_center': (y_positions[s]+y_positions[e])/2,
            'width_mm': y_positions[e]-y_positions[s],
            'height':   np.max(inverted[s:e+1]) if e > s else inverted[s]
        })
    return regions

def merge_nearby_regions(regions, merge_gap=10, complete_width=45):
    if not regions:
        return []
    regions.sort(key=lambda r: r['y_start'])
    merged = [regions[0].copy()]
    for r in regions[1:]:
        if (merged[-1]['width_mm'] < complete_width and r['y_start'] - merged[-1]['y_end'] <= merge_gap):
            merged[-1]['y_end']   = r['y_end']
            merged[-1]['height']  = max(merged[-1]['height'], r['height'])
        else:
            merged[-1]['y_center'] = (merged[-1]['y_start']+merged[-1]['y_end'])/2
            merged[-1]['width_mm'] = merged[-1]['y_end']-merged[-1]['y_start']
            merged.append(r.copy())
    merged[-1]['y_center'] = (merged[-1]['y_start']+merged[-1]['y_end'])/2
    merged[-1]['width_mm'] = merged[-1]['y_end']-merged[-1]['y_start']
    return merged

def find_stoppers(regions, player_color):
    req_width = 35 if player_color == 'dark' else 10
    valid = sorted([r for r in regions if r['width_mm'] >= req_width], key=lambda p: p['height'], reverse=True)
    if len(valid) < 2:
        return None, None
    return tuple(sorted(valid[:2], key=lambda p: p['y_center']))

def detect_rod(intensities, y_positions, rod_num, config):
    inverted  = 255 - intensities
    threshold = np.percentile(inverted, 50) + 30
    regions   = find_peaks_above_threshold(inverted, y_positions, threshold)
    merged    = merge_nearby_regions(regions)
    top, bottom = find_stoppers(merged, config['player_color'])
    if not top or not bottom:
        return []
    if config['offsets_bottom'] is None:
        return [(top['y_center']+bottom['y_center'])/2]
    if FIELD_HEIGHT_MM - bottom['y_end'] <= top['y_start']:
        poi, offsets = bottom['y_start'], config['offsets_bottom']
    else:
        poi, offsets = top['y_end'], config['offsets_top']
    return sorted([poi + o for o in offsets])


print("Detection functions defined.")

In [ ]:
K_new = np.load(K_NEW_PATH)
map1 = np.load(MAP1_PATH)
map2 = np.load(MAP2_PATH)
print("Calibration loaded.")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
dt = 1.0 / fps
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {total_frames} frames at {fps:.1f} fps")

H_matrix, rvec_global, tvec_global = None, None, None
for attempt in range(100):
    ret, frame = cap.read()
    if not ret:
        break
    img_undist = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)
    H_matrix, rvec_global, tvec_global = compute_homography_and_pose(img_undist, K_new)
    if H_matrix is not None:
        print(f"Homography from frame {attempt + 1}")
        break

kf = cv2.KalmanFilter(4, 2)
kf.measurementMatrix = np.array([[1,0,0,0],[0,1,0,0]], np.float32)
kf.transitionMatrix = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], np.float32)
kf.processNoiseCov = np.eye(4, dtype=np.float32) * 0.03
print("Ready!")

In [ ]:
# INTERACTIVE VIEWER
out_w, out_h = int(FIELD_WIDTH_MM + 2*PAD), int(FIELD_HEIGHT_MM + 2*PAD)
INFO_SIZE    = 500

tracking_results  = {}
hit_frames        = set()

# Goal sets - frame number keyed so replay never double counts
# Left  goal (ball x < 50)                 = Black goal  -> WHITE scored
# Right goal (ball x > FIELD_WIDTH_MM - 50) = White goal -> BLACK scored
goal_frames_black = set()
goal_frames_white = set()

prev_status            = "Searching"
prev_x, prev_y, prev_v = None, None, 0.0
prev_vx, prev_vy       = 0.0, 0.0
prev_frame_num         = -1
goal_counter           = 0

display_hit_frame  = None
display_hit_player = "None"
display_hit_reason = "None"
hit_display_until  = 0.0
last_onfield_x = None
last_onfield_y = None

paused = False

# Entry - exit hit detection logic
zone_states = {}

cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
reset_stabilization()

cv2.namedWindow("Foosball Tracker", cv2.WINDOW_NORMAL)
cv2.namedWindow("Info Panel",       cv2.WINDOW_NORMAL)


def init_zone_states(all_player_positions):
    states = {}
    for rod_num, positions in all_player_positions.items():
        for idx in range(len(positions)):
            pid = get_player_id(rod_num, idx)
            states[pid] = {
                'active':           False,
                'entry_vx':         0.0,
                'entry_vy':         0.0,
                'entry_speed':      0.0,
                'entry_frame':      -1,
                'frames_inside':    0,
                'min_speed_inside': 999.0,
                'was_stopped':      False
            }
    return states


def reset_tracking_state():
    global prev_x, prev_y, prev_v, prev_vx, prev_vy
    global prev_frame_num, goal_counter, prev_status
    global display_hit_frame, display_hit_player, display_hit_reason, hit_display_until
    global zone_states
    global last_onfield_x, last_onfield_y
    prev_x, prev_y, prev_v = None, None, 0.0
    prev_vx, prev_vy       = 0.0, 0.0
    prev_frame_num         = -1
    goal_counter           = 0
    prev_status            = "Searching"
    display_hit_frame      = None
    display_hit_player     = "None"
    display_hit_reason     = "None"
    hit_display_until      = 0.0
    zone_states            = {}   # will be rebuilt on first frame


def process_frame(frame, frame_num):
    global prev_x, prev_y, prev_v, prev_vx, prev_vy, prev_frame_num
    global goal_counter, prev_status, goal_frames_black, goal_frames_white
    global display_hit_frame, display_hit_player, display_hit_reason, hit_display_until
    global zone_states
    global last_onfield_x, last_onfield_y

    timestamp = frame_num * dt
    frame_hit_player = "No"
    frame_hit_reason = "No"

    predicted  = kf.predict()
    pred_x_raw = int(predicted[0][0])
    pred_y_raw = int(predicted[1][0])

    img_undist = cv2.remap(frame, map1, map2, cv2.INTER_LINEAR)
    warped     = cv2.warpPerspective(img_undist, H_matrix, (out_w, out_h))
    display    = warped.copy()

    # Ball detection
    hsv  = cv2.cvtColor(warped, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_RED, UPPER_RED)
    mask = cv2.erode(mask,  None, iterations=1)
    mask = cv2.dilate(mask, None, iterations=2)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    measured_pt = None
    detected_x_mm = detected_y_mm = None
    for cnt in contours:
        ((x, y), radius) = cv2.minEnclosingCircle(cnt)
        if 5 < radius < 35:
            measured_pt   = np.array([[np.float32(x)], [np.float32(y)]])
            detected_x_mm = int(x) - PAD
            detected_y_mm = int(y) - PAD
            break

    # Status logic
    status = "Searching"
    log_x = log_y = None

    if measured_pt is not None:
        kf.correct(measured_pt)
        
        if 0 <= detected_x_mm <= FIELD_WIDTH_MM and 0 <= detected_y_mm <= FIELD_HEIGHT_MM:
            # Ball is ON FIELD
            goal_counter = 0
            status, log_x, log_y = "On Field", detected_x_mm, detected_y_mm
            last_onfield_x, last_onfield_y = detected_x_mm, detected_y_mm
        else:
            # Ball is OUTSIDE field ("Wall")
            # Check if near goal area
            near_left_goal = detected_x_mm < 0 and abs(detected_y_mm - GOAL_Y_CENTER) < GOAL_TOLERANCE
            near_right_goal = detected_x_mm > FIELD_WIDTH_MM and abs(detected_y_mm - GOAL_Y_CENTER) < GOAL_TOLERANCE
            
            if near_left_goal or near_right_goal:
                # Ball in goal area - count towards goal
                goal_counter += 1
                if goal_counter > 5:
                    status, log_x, log_y = "Goal", "Goal", "Goal"
                else:
                    status, log_x, log_y = "Wall", "Wall", "Wall"
            else:
                # Wall but not near goal - reset
                goal_counter = 0
                status, log_x, log_y = "Wall", "Wall", "Wall"
    else:
        # Ball not detected
        pred_x_mm = pred_x_raw - PAD
        pred_y_mm = pred_y_raw - PAD
        
        # Check if last known position was near goal
        if (last_onfield_x is not None and
                (last_onfield_x < 50 or last_onfield_x > FIELD_WIDTH_MM - 50) and
                abs(last_onfield_y - GOAL_Y_CENTER) < GOAL_TOLERANCE):
            goal_counter += 1
            if goal_counter > 5:
                status, log_x, log_y = "Goal", "Goal", "Goal"
        
        if status != "Goal":
            if 0 <= pred_x_mm <= FIELD_WIDTH_MM and 0 <= pred_y_mm <= FIELD_HEIGHT_MM:
                status, log_x, log_y = "Occluded", pred_x_mm, pred_y_mm

    # Goal counting
    if status == "Goal" and prev_status != "Goal":
        if prev_x is not None and isinstance(prev_x, int):
            if prev_x < 50:
                goal_frames_white.add(frame_num)
            elif prev_x > FIELD_WIDTH_MM - 50:
                goal_frames_black.add(frame_num)
    prev_status = status

    goals_black = len(goal_frames_black)
    goals_white = len(goal_frames_white)

    # Ball display position
    ball_x_mm = ball_y_mm = None
    ball_disp_x = ball_disp_y = None

    if isinstance(log_x, int):
        ball_x_mm, ball_y_mm     = log_x, log_y
        ball_disp_x, ball_disp_y = log_x + PAD, log_y + PAD
    elif measured_pt is not None:
        ball_disp_x = int(measured_pt[0][0])
        ball_disp_y = int(measured_pt[1][0])
    elif status == "Occluded":
        ball_disp_x, ball_disp_y = pred_x_raw, pred_y_raw

    if ball_disp_x:
        color = (0, 255, 0) if measured_pt is not None else (0, 255, 255)
        cv2.circle(display, (int(ball_disp_x), int(ball_disp_y)), 15, color, 2)

    # Physics 
    curr_v = accel = curr_vx = curr_vy = 0.0
    if isinstance(log_x, int) and prev_x is not None and isinstance(prev_x, int):
        dx, dy  = log_x - prev_x, log_y - prev_y
        curr_v  = math.sqrt(dx**2 + dy**2) / 1000.0 / dt
        curr_vx = dx / 1000.0 / dt
        curr_vy = dy / 1000.0 / dt
        accel   = (curr_v - prev_v) / dt

    if isinstance(log_x, int):
        prev_x, prev_y, prev_v = log_x, log_y, curr_v

    #  Players and hit zones
    all_player_positions = {}
    cv2.rectangle(display, (PAD, PAD), (PAD + FIELD_WIDTH_MM, PAD + FIELD_HEIGHT_MM), (255, 0, 0), 2)

    for rod_num, config in ROD_CONFIG.items():
        points, y_pos = project_rod_line_to_warped(config['x'], rvec_global, tvec_global, K_new, H_matrix)
        rod_color = (0, 0, 255) if config['team'] == 'Black' else (255, 255, 0)
        for i in range(0, len(points) - 1, 2):
            cv2.line(display, points[i], points[i + 1], rod_color, 1)

        intensities = extract_intensities_along_line(warped, points)
        smoothed    = apply_bilateral_filter_1d(intensities)
        player_pos  = smart_stabilize(rod_num, detect_rod(smoothed, y_pos, rod_num, config))
        all_player_positions[rod_num] = player_pos

        for pidx, py in enumerate(player_pos):
            idx = np.argmin(np.abs(y_pos - py))
            if 0 <= idx < len(points):
                cv2.circle(display, points[idx], 10, (0, 255, 0), 2)
                hw, hh = HIT_ZONE_WIDTH // 2, HIT_ZONE_HEIGHT // 2
                z1 = (int(config['x'] + PAD - hw), int(py + PAD - hh))
                z2 = (int(config['x'] + PAD + hw), int(py + PAD + hh))
                zc = (0, 0, 255) if check_ball_in_hit_zone(ball_x_mm, ball_y_mm, config['x'], py) else (0, 255, 0)
                cv2.rectangle(display, z1, z2, zc, 2)
                cv2.putText(display, get_player_id(rod_num, pidx),
                            (z1[0], z1[1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)

    # Initialise zone_states on first frame or after reset
    if not zone_states:
        zone_states.update(init_zone_states(all_player_positions))

    current_hit_zone = find_ball_in_hit_zones(ball_x_mm, ball_y_mm, all_player_positions)

    for rod_num, config in ROD_CONFIG.items():
        rx = config['x']
        positions = all_player_positions.get(rod_num, [])
        for pidx, py in enumerate(positions):
            pid = get_player_id(rod_num, pidx)

            # Ensure state entry exists
            if pid not in zone_states:
                zone_states[pid] = {
                    'active': False, 'entry_vx': 0.0, 'entry_vy': 0.0,
                    'entry_speed': 0.0, 'entry_frame': -1, 'frames_inside': 0,
                    'min_speed_inside': 999.0, 'was_stopped': False
                }

            in_zone_now = check_ball_in_hit_zone(ball_x_mm, ball_y_mm, rx, py)
            was_active  = zone_states[pid]['active']

            if not was_active and in_zone_now:
                zone_states[pid]['active']           = True
                zone_states[pid]['entry_vx']         = curr_vx
                zone_states[pid]['entry_vy']         = curr_vy
                zone_states[pid]['entry_speed']      = curr_v
                zone_states[pid]['entry_frame']      = frame_num
                zone_states[pid]['frames_inside']    = 1
                zone_states[pid]['min_speed_inside'] = curr_v
                zone_states[pid]['was_stopped']      = curr_v < STOPPED_THRESHOLD

            elif was_active and in_zone_now:
                zone_states[pid]['frames_inside'] += 1
                
                # Track minimum speed
                if curr_v < zone_states[pid]['min_speed_inside']:
                    zone_states[pid]['min_speed_inside'] = curr_v
                
                # Check if ball was ever stopped
                if curr_v < STOPPED_THRESHOLD:
                    zone_states[pid]['was_stopped'] = True

            elif was_active and not in_zone_now:
                zone_states[pid]['active'] = False

                entry_speed   = zone_states[pid]['entry_speed']
                entry_vx      = zone_states[pid]['entry_vx']
                entry_vy      = zone_states[pid]['entry_vy']
                exit_speed    = curr_v
                exit_vx       = curr_vx
                exit_vy       = curr_vy
                frames_inside = zone_states[pid]['frames_inside']
                min_speed     = zone_states[pid]['min_speed_inside']
                was_stopped   = zone_states[pid]['was_stopped']
                entry_frame   = zone_states[pid]['entry_frame']

                hit_type = None

                # Calculate direction change
                direction_change = 0.0
                if entry_speed > STOPPED_THRESHOLD and exit_speed > STOPPED_THRESHOLD:
                    cos_a = max(-1, min(1,
                        (entry_vx * exit_vx + entry_vy * exit_vy) /
                        (entry_speed * exit_speed)))
                    direction_change = math.degrees(math.acos(cos_a))

                #CLASSIFICATION LOGIC
                
                if frames_inside < MIN_FRAMES_CONTROL:
                    # quick contact           
                    if direction_change > BLOCK_ANGLE:
                        hit_type = f"Block {direction_change:.0f}deg"
                    elif direction_change > MIN_DIRECTION_CHANGE:
                        hit_type = f"Brush {direction_change:.0f}deg"
                    # else: No Hit (ball passed through without meaningful contact)
                
                else:
                    # controlled contact
                    
                    if was_stopped:
                        # Ball was stopped at some point
                        if exit_speed >= FAST_EXIT_THRESHOLD:
                            hit_type = "Strong Kick"
                        elif exit_speed >= SLOW_EXIT_THRESHOLD:
                            hit_type = "Weak Kick"
                        else:
                            hit_type = "Stop (No Kick)"
                    else:
                        # Ball was never stopped 
                        if exit_speed >= FAST_EXIT_THRESHOLD:
                            hit_type = "Strong Dribble & Kick"
                        elif exit_speed >= SLOW_EXIT_THRESHOLD:
                            hit_type = "Weak Dribble & Kick"
                        else:
                            hit_type = "Dribble (No Kick)"

                # LOG THE HIT
                if hit_type and frame_num not in hit_frames:
                    hit_frames.add(frame_num)
                    display_hit_frame   = frame_num
                    display_hit_player  = pid
                    display_hit_reason  = hit_type
                    hit_display_until   = time.time() + 2.0
                    
                    # Store for logging
                    frame_hit_player = pid
                    frame_hit_reason = hit_type
    
    prev_vx, prev_vy = curr_vx, curr_vy
    prev_frame_num   = frame_num

    # Clear hit display after 2 real seconds
    if time.time() > hit_display_until:
        display_hit_frame  = None
        display_hit_player = "None"
        display_hit_reason = "None"

    tracking_results[frame_num] = {
        'Frame':          frame_num,
        'Timestamp':      round(timestamp, 4),
        'X':              log_x,
        'Y':              log_y,
        'Speed_m_s':      round(curr_v, 3),
        'Accel_m_s2':     round(accel, 3),
        'Status':         status,
        'Hit_Zone':       current_hit_zone,
        'Hit_Successful': "Yes" if frame_hit_player != "No" else "No",
        'Hit_Player':     frame_hit_player,
        'Hit_Reason':     frame_hit_reason,
        'Hit_Category':   "Stop" if "Stop" in str(frame_hit_reason) else 
                          "Dribble" if "Dribble" in str(frame_hit_reason) else
                          "Brush" if "Brush" in str(frame_hit_reason) else
                          "Block" if "Block" in str(frame_hit_reason) else "No"
    }

    # Info panel
    info = np.zeros((INFO_SIZE, INFO_SIZE, 3), dtype=np.uint8)
    info[:] = (30, 30, 30)
    cv2.line(info, (250, 15), (250, 485), (80, 80, 80), 1)

    # Left column
    LX, y = 15, 35
    cv2.putText(info, "SCORE",               (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 40
    cv2.putText(info, f"Black: {goals_black}",(LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (100,100,255), 2); y += 30
    cv2.putText(info, f"White: {goals_white}",(LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,100), 2); y += 50

    cv2.putText(info, "BALL TRACKING",        (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 35
    cv2.putText(info, f"Frame: {frame_num}",  (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1); y += 25
    cv2.putText(info, f"Time: {timestamp:.3f}s",(LX,y),cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1); y += 25
    sc = (0,255,0) if status=="On Field" else (0,255,255) if status=="Occluded" else (0,0,255)
    cv2.putText(info, f"Status: {status}",    (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, sc, 2); y += 50

    cv2.putText(info, "POSITION",             (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 35
    if isinstance(log_x, int):
        cv2.putText(info, f"X: {log_x} mm",   (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1); y += 25
        cv2.putText(info, f"Y: {log_y} mm",   (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)
    else:
        cv2.putText(info, "X: N/A",           (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100,100,100), 1); y += 25
        cv2.putText(info, "Y: N/A",           (LX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100,100,100), 1)

    # Right column
    RX, y = 265, 35
    cv2.putText(info, "PHYSICS",              (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 35
    cv2.putText(info, f"Speed: {curr_v:.2f} m/s",(RX,y),cv2.FONT_HERSHEY_SIMPLEX,0.55,(200,200,200),1); y += 25
    cv2.putText(info, f"Accel: {accel:.1f} m/s2",(RX,y),cv2.FONT_HERSHEY_SIMPLEX,0.55,(200,200,200),1); y += 50

    cv2.putText(info, "HIT DETECTION",        (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 35
    zc = (0,255,255) if current_hit_zone != "None" else (100,100,100)
    cv2.putText(info, f"Zone: {current_hit_zone}",     (RX,y),cv2.FONT_HERSHEY_SIMPLEX,0.55,zc,1); y += 25
    cv2.putText(info, f"Total Hits: {len(hit_frames)}",(RX,y),cv2.FONT_HERSHEY_SIMPLEX,0.55,(200,200,200),1); y += 30

    if display_hit_frame is not None:
        # Color code by hit category
        if "Stop" in display_hit_reason:
            hit_color = (0, 255, 255)    # Yellow for Stop
        elif "Dribble" in display_hit_reason:
            hit_color = (255, 165, 0)    # Orange for Dribble
        elif "Block" in display_hit_reason:
            hit_color = (0, 0, 255)      # Red for Block
        elif "Brush" in display_hit_reason:
            hit_color = (255, 255, 0)    # Cyan for Brush
        else:
            hit_color = (0, 255, 0)      # Green default
        
        cv2.putText(info, "HIT!",                          (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.7,  hit_color, 2); y += 28
        cv2.putText(info, f"Frame:  {display_hit_frame}",  (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5,  hit_color, 1); y += 22
        cv2.putText(info, f"Player: {display_hit_player}", (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5,  hit_color, 1); y += 22
        cv2.putText(info, f"Type:   {display_hit_reason}", (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.5,  hit_color, 1); y += 22
    else:
        y += 94

    y += 10
    cv2.putText(info, "CONTROLS",             (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255,255,255), 2); y += 30
    cv2.putText(info, "[SPACE] Pause/Play",   (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150,150,150), 1); y += 20
    cv2.putText(info, "[Q] Quit & Save",      (RX, y), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (150,150,150), 1)

    return display, info


# Main loop
ret, frame = cap.read()
current_frame = 0

while ret:
    display, info = process_frame(frame, current_frame)
    cv2.imshow("Foosball Tracker", display)
    cv2.imshow("Info Panel",       info)

    key = cv2.waitKey(30 if not paused else 0) & 0xFF

    if key == ord('q'):
        break
    elif key == ord(' '):
        paused = not paused
    elif not paused:
        ret, frame = cap.read()
        if ret:
            current_frame += 1
        else:
            # Video ended
            break

cv2.destroyAllWindows()

sorted_results = [tracking_results[k] for k in sorted(tracking_results.keys())]
pd.DataFrame(sorted_results).to_csv(OUTPUT_CSV, index=False)
print(f"\nSaved to {OUTPUT_CSV}")
print(f"Frames: {len(sorted_results)}, Hits: {len(hit_frames)}")
print(f"Score - Black: {len(goal_frames_black)}, White: {len(goal_frames_white)}")


In [ ]:
cap.release()
print("Done!")